In [1]:
pip install nltk

Note: you may need to restart the kernel to use updated packages.


In [2]:
import nltk
from nltk.chat.util import Chat, reflections
from nltk.sentiment import SentimentIntensityAnalyzer 

In [3]:
nltk.download('punkt')
nltk.download('vader_lexicon')

[nltk_data] Downloading package punkt to C:\Users\shafi
[nltk_data]     laptop\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package vader_lexicon to C:\Users\shafi
[nltk_data]     laptop\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [4]:
pairs = [

    # =========================
    # GREETINGS
    # =========================
    
    [r"(?i)\b(hello|hi|hey|assalam|good morning|good afternoon|good evening)\b",
     ["Hello! Welcome to City Hospital. How can I assist you today? 🏥",
      "Hi there! I'm your Hospital Assistant. Ask me about location, doctors, appointments, or emergency services!",
      "Assalam-o-Alaikum! Welcome to City Hospital. How may I help you today?"]],

    # =========================
    # HOSPITAL NAME - CATCHES ALL NAME QUERIES
    # =========================
    
    [r"(?i).*\b(name|what.*call|which.*hospital|your name)\b.*",
     ["Welcome to City Hospital - One of the leading healthcare providers with 500+ beds and 50+ specialties. Established in 1995.",
      "This is City Hospital, a NABH and JCI accredited multi-specialty hospital serving the community for over 25 years.",
      "You are contacting City Hospital - Your trusted healthcare partner with 24/7 emergency services and world-class facilities."]],

    # =========================
    # LOCATION - CATCHES ALL LOCATION QUERIES (LONG & SHORT)
    # =========================
    
    [r"(?i).*\b(where.*located|where.*hospital|address|location|where.*find|how.*reach|how.*get.*there|direction)\b.*",
     ["City Hospital is located at 123 Main Boulevard, City Center, near Central Park and opposite Metro Station.",
      "Our address: 123 Main Boulevard, City Center. Landmark: Next to Central Park, Opposite Metro Station. Free parking available.",
      "You can find us at 123 Main Boulevard, City Center. We're easily accessible from all major areas."]],

    # Combined name + location
    [r"(?i).*\b(name.*address|name.*location|where.*and.*name|address.*and.*name)\b.*",
     ["This is City Hospital located at 123 Main Boulevard, City Center (Near Central Park). We're a 500-bed multi-specialty hospital.",
      "Welcome to City Hospital! Address: 123 Main Boulevard, City Center. Landmark: Opposite Metro Station, Near Central Park."]],

    # =========================
    # EMERGENCY - CATCHES ALL EMERGENCY QUERIES
    # =========================
    
    [r"(?i).*\b(emergency|urgent|critical|accident|1122)\b.*",
     ["Emergency Department: Ground floor, separate entrance. Open 24/7! Call: 1122 or 042-XXXXXXX",
      "24/7 Emergency services available! Trauma, Cardiac, Stroke, Pediatric emergency. Emergency hotline: 1122",
      "Our Emergency is open 24/7/365. Average wait time: 15 minutes. Ambulance: 1122"]],

    # Emergency number specifically
    [r"(?i).*\b(emergency.*number|emergency.*contact|ambulance.*number|helpline|phone.*emergency)\b.*",
     ["EMERGENCY NUMBERS: Ambulance: 1122, Hospital Emergency: 042-XXXXXXX, Chest Pain: 042-HEART-1",
      "Emergency contacts: 1122 (Ambulance), 042-XXXXXXX (Hospital ER). Save these numbers!"]],

    # =========================
    # APPOINTMENT BOOKING - CATCHES ALL BOOKING QUERIES
    # =========================
    
    [r"(?i).*\b(appointment|book.*doctor|schedule.*visit|OPD.*booking|see.*doctor|want.*appointment|need.*appointment)\b.*",
     ["Book Appointment: 1) Online: www.cityhospital.com/book 2) Call: 042-XXXXXXX 3) WhatsApp: +92-XXX-XXXXXXX 4) Walk-in",
      "You can book by: Online portal, Phone (042-XXXXXXX), WhatsApp, or visit reception. Which is easiest for you?",
      "Booking options: Video consultation, In-person OPD, or Home visit. Call 042-XXXXXXX or book online."]],

    # =========================
    # DOCTOR FEES - CATCHES ALL FEE QUERIES
    # =========================
    
    [r"(?i).*\b(fee|charges|cost|price|payment|rate|kitna.*lagay|how.*much)\b.*",
     ["Consultation Fees: General Physician Rs. 500, Specialist Rs. 1000-1500, Super-specialist Rs. 2000.",
      "Doctor charges: General Rs. 500, Specialist Rs. 1000-1500, Professor Rs. 2500. Follow-up 50% off within 7 days.",
      "Fees vary by specialty: General Rs. 500, Heart/Brain specialists Rs. 1500-2000. Room charges: General ward Rs. 2000/day, Private Rs. 5000/day."]],

    # =========================
    # TIMINGS - CATCHES ALL TIME QUERIES
    # =========================
    
    [r"(?i).*\b(timing|time|hours|open|close|schedule|when.*open|what.*time)\b.*",
     ["OPD Timings: Morning 9 AM - 1 PM, Evening 5 PM - 8 PM (Mon-Sat). Sunday: Emergency only.",
      "Outpatient hours: 9 AM - 1 PM and 5 PM - 8 PM. Emergency: 24/7. Pharmacy: 24/7. Lab: 7 AM - 9 PM.",
      "We are open 24/7 for emergency. Regular OPD: 9 AM-1 PM, 5 PM-8 PM. Lab: 7 AM-9 PM."]],

    # Visiting hours specifically
    [r"(?i).*\b(visiting.*hour|visit.*time|meet.*patient|guest.*hour|family.*visit|when.*visit)\b.*",
     ["Visiting Hours: General wards 4 PM - 7 PM, Private rooms 8 AM - 8 PM, ICU 4 PM - 5 PM (2 visitors only).",
      "You can visit: General wards 4-7 PM, Private rooms flexible 8 AM-8 PM, ICU only 4-5 PM (10 mins, 2 visitors)."]],

    # =========================
    # DEPARTMENTS - CATCHES ALL DEPARTMENT QUERIES
    # =========================
    
    [r"(?i).*\b(department|service|specialty|facility|what.*offer|which.*department|available.*service)\b.*",
     ["Departments: Emergency, Cardiology, Neurology, Orthopedics, Pediatrics, Maternity, Surgery, ICU, Radiology, Lab, Pharmacy.",
      "Our services: 24/7 Emergency, Heart care, Brain/Neurology, Bones/Joints, Children care, Maternity, All surgeries, ICU, Diagnostics.",
      "We offer: Emergency, Heart, Brain, Bones, Kids, Maternity, Surgery, ICU, X-ray/MRI, Blood tests, Pharmacy - all under one roof."]],

    # ICU specific
    [r"(?i).*\b(ICU|intensive.*care|critical.*care|ventilator|life.*support)\b.*",
     ["ICU: 50 beds total - Medical ICU, Surgical ICU, Cardiac ICU, Neonatal ICU, Pediatric ICU. 24/7 intensivist coverage.",
      "Critical Care: 50 beds with ventilators, monitors, dialysis support. 1:1 nursing ratio. Visiting: 4-5 PM only."]],

    # Pharmacy
    [r"(?i).*\b(pharmacy|medicine|drug.*store|medical.*store|dawai)\b.*",
     ["Pharmacy: 24/7 open! All medicines available. Home delivery in local area. Insurance cashless accepted.",
      "Yes! 24/7 in-house pharmacy. Generic and branded medicines. Home delivery 8 AM-10 PM."]],

    # Lab/Tests
    [r"(?i).*\b(lab|test|blood.*test|pathology|diagnostic|report|x-ray|MRI|CT|ultrasound|scan)\b.*",
     ["Laboratory: All blood tests, X-ray, Ultrasound, CT scan, MRI. 24/7 emergency lab. Home sample collection 6 AM-10 PM.",
      "Diagnostics: Blood tests, X-ray, MRI, CT scan, Ultrasound. Most reports same day. Call for home sample collection."]],

    # =========================
    # SPECIALIZED HOSPITALS - CATCHES ALL
    # =========================
    
    [r"(?i).*\b(children.*hospital|kids.*hospital|pediatric|baby.*hospital|child.*specialist)\b.*",
     ["Children's Hospital: 45 New Town. 200 beds, NICU, Pediatric ICU, Vaccination, Child development. 24/7 pediatric emergency.",
      "Kids Care: New Town location. All child specialists, vaccination center, play area, child-friendly environment."]],

    [r"(?i).*\b(heart.*hospital|cardiac|cardiology|heart.*attack|chest.*pain|heart.*care)\b.*",
     ["Heart Care Institute: 123 Main Boulevard. 100 cardiac beds, 3 Cath labs, Bypass, Angioplasty 24/7. 98% success rate.",
      "Cardiac Center: Emergency angioplasty within 90 minutes. Chest pain hotline: 042-HEART-1."]],

    [r"(?i).*\b(cancer.*hospital|oncology|chemotherapy|tumor|radiation.*therapy|cancer.*treatment)\b.*",
     ["Cancer Institute: Medical City. Chemotherapy, Radiation therapy, Surgery, Bone marrow transplant. Free screening camps monthly.",
      "Oncology center: All cancer treatments including PET-CT, Daycare chemo, Palliative care."]],

    [r"(?i).*\b(maternity|delivery|pregnancy|women.*hospital|gynecology|baby.*delivery|obstetrics)\b.*",
     ["Women's Hospital: Garden Road. 150 maternity beds, LDR rooms, NICU, IVF center. Painless delivery available.",
      "Maternity services: Normal delivery, C-section, High-risk pregnancy care, Antenatal classes. 24/7 labor rooms."]],

    [r"(?i).*\b(bone.*hospital|orthopedic|joint.*replacement|fracture|spine.*surgery|sports.*medicine)\b.*",
     ["Bone & Joint Hospital: Stadium Road. Knee/hip replacement, Arthroscopy, Spine surgery, 24/7 fracture clinic.",
      "Orthopedics: Joint replacement center, Sports medicine, Physiotherapy. 95% success rate for joint surgeries."]],

    [r"(?i).*\b(eye.*hospital|eye.*care|ophthalmology|LASIK|cataract.*surgery|vision.*center)\b.*",
     ["Eye Institute: 12 Main Market. Cataract surgery, LASIK, Retinal surgery, Glaucoma treatment. Free eye camps every Sunday.",
      "Vision Care Center: Advanced eye care with OCT, Pediatric ophthalmology. Located at main campus, 2nd floor."]],

    [r"(?i).*\b(dental.*hospital|dentist|orthodontics|root.*canal|oral.*surgery|teeth)\b.*",
     ["Dental Hospital: 34 Commercial Area. Implants, Braces, Root canal, Oral surgery, Cosmetic dentistry. Digital X-ray.",
      "Smile Care Center: 5th Floor, City Hospital main campus. All dental specialties, 24/7 dental trauma care."]],

    [r"(?i).*\b(mental.*hospital|psychiatry|psychology|rehab|depression|addiction|drug.*treatment)\b.*",
     ["Mind Care Institute: 67 Peace Valley. Depression, Anxiety, Addiction rehab, De-addiction programs. Confidential care.",
      "Psychiatry Department: 3rd Floor, Main Hospital. Inpatient/outpatient, Therapy, Crisis intervention. 24/7 helpline."]],

    # =========================
    # AMBULANCE
    # =========================
    
    [r"(?i).*\b(ambulance|emergency.*vehicle|transport|medical.*transport)\b.*",
     ["Ambulance: 24/7 service. Call 1122. City: Rs. 500-1000, Outstation: Rs. 50/km. Advanced life support available.",
      "Yes! 24/7 ambulance with life support equipment. Call 1122 or 042-XXXXXXX."]],

    # =========================
    # INSURANCE
    # =========================
    
    [r"(?i).*\b(insurance|health.*card|cashless|TPA|panel|coverage|sehat.*card|health.*scheme)\b.*",
     ["We accept all major insurance: 50+ companies, Government schemes (Sehat Card, Ayushman Bharat), Corporate plans. Cashless facility available.",
      "Insurance accepted: All major TPAs, Health cards, Government schemes. Pre-authorization within 2-4 hours."]],

    # =========================
    # BED AVAILABILITY
    # =========================
    
    [r"(?i).*\b(bed.*available|room.*vacant|admission.*possible|ICU.*bed|bed.*empty|room.*available)\b.*",
     ["Current status: General ward - Available, Private rooms - Limited, ICU - 3 beds free. Call 042-XXXXXXX ext. 101 for real-time status.",
      "Bed availability changes hourly. Please call 042-XXXXXXX (ext. 101) for current status or check online."]],

    # =========================
    # PARKING & FACILITIES
    # =========================
    
    [r"(?i).*\b(parking|car.*park|vehicle.*park|where.*park)\b.*",
     ["FREE parking for 200 vehicles! Basement + Ground level. Valet service available. Security guarded 24/7.",
      "Yes! Free parking for patients and visitors. 200+ spaces, wheelchair accessible, EV charging stations."]],

    [r"(?i).*\b(wifi|internet|password|network|wireless)\b.*",
     ["Free WiFi: Network 'CityHospital-Guest', Password: patientcare123. Available in all areas.",
      "WiFi available: 'CityHospital-Guest', Password: patientcare123. High-speed, unlimited."]],

    [r"(?i).*\b(canteen|food|cafeteria|meal|restaurant|eating)\b.*",
     ["Canteen: 24/7 open. Vegetarian and non-vegetarian meals. Patient diet as per doctor. Affordable prices.",
      "Food services: 24/7 cafeteria, Patient kitchen, Room service. Special diets for diabetes, heart, kidney patients."]],

    # =========================
    # FEEDBACK & COMPLAINTS
    # =========================
    
    [r"(?i).*\b(complaint|feedback|suggestion|problem|issue|grievance|dissatisfied|unhappy)\b.*",
     ["Feedback: Patient relations officer (Room 101), Suggestion boxes, Toll-free: 1800-XXX-XXXX. Resolution within 48 hours.",
      "Sorry for inconvenience. Please call 1800-XXX-XXXX or visit Room 101. We resolve complaints within 48 hours."]],

    # =========================
    # COVID/VACCINATION
    # =========================
    
    [r"(?i).*\b(covid|corona|vaccine|vaccination|booster|PCR|RT.*PCR|covid.*test)\b.*",
     ["COVID services: RT-PCR testing, Vaccination (all types), Booster doses. Separate COVID OPD. Call for home testing.",
      "Vaccination center: All COVID vaccines available. Walk-in accepted. International travel certificates provided."]],

    # =========================
    # GOODBYE
    # =========================
    
    [r"(?i).*\b(bye|goodbye|take.*care|thank|thanks)\b.*",
     ["Goodbye! Stay healthy and take care! 😊 Visit again if needed.",
      "Thank you! Wishing you good health. We're here 24/7 if you need us."]],

    # =========================
    # HELP
    # =========================
    
    [r"(?i).*\b(help|what.*can.*do|how.*help|assist|support)\b.*",
     ["I can help you with:\n🏥 Hospital location & name\n📞 Emergency numbers\n📅 Book appointments\n💰 Doctor fees & charges\n⏰ Timings & visiting hours\n🩺 Departments & services\n🚑 Ambulance service\n💳 Insurance & payments\n\nJust ask what you need!"]],

    # =========================
    # FALLBACK - LAST RESORT
    # =========================
    
    [r"(?i).+",
     ["Sorry, I didn't understand. Try asking:\n• 'Where is hospital located?'\n• 'Emergency number'\n• 'Book appointment'\n• 'Doctor fees'\n• 'Timings'\n• 'Departments'\n\nOr type 'help' to see all options!",
      
      "Main samjha nahi. Aap yeh try karain:\n• 'Hospital kahan hai?' (Location)\n• 'Emergency number'\n• 'Appointment chahiye'\n• 'Doctor fees'\n• 'Timings'\n\nYa 'help' likhain!"]],

    # Empty input
    [r"^$",
     ["Please type your question. I can help with: Location, Emergency, Appointment, Fees, Timings, Departments. Or type 'help' for all options.",
      "Kuch likhain please. Mein hospital ke bare mein kya bata sakta hoon?"]]
]



In [5]:
chatbot = Chat(pairs, reflections)
sia = SentimentIntensityAnalyzer()

In [6]:
def analyze_sentiment(text):
    score = sia.polarity_scores(text)

    if score['compound'] >= 0.05:
        return "Positive 😊"
    elif score['compound'] <= -0.05:
        return "Negative 😡"
    else:
        return "Neutral 😐"

In [7]:
def chatbot_app():
    print("\n🏥 Welcome to Hospital Chatbot")
    print("Type 'sentiment' to analyze mood")
    print("Type 'exit' to quit\n")

    while True:
        user_input = input("You: ")

        if user_input.lower() == "exit":
            print("Bot: Goodbye! Stay healthy 😊")
            break

        elif user_input.lower() == "sentiment":
            text = input("Enter sentence: ")
            print("Bot:", analyze_sentiment(text))

        else:
            # Sentiment-aware response
            mood = analyze_sentiment(user_input)

            if "Negative" in mood:
                print("Bot: I'm sorry to hear that 😔")

            response = chatbot.respond(user_input)

            if response:
                print("Bot:", response)
            else:
                print("Bot: Sorry, I didn't understand that.")

In [ ]:
import gradio as gr

def chat_interface(user_input, history):
    if not user_input.strip():
        return "Please type your question."

    mood = analyze_sentiment(user_input)
    prefix = ""
    if "Negative" in mood:
        prefix = "I'm sorry to hear that. Let me help you.\n\n"

    response = chatbot.respond(user_input)
    return prefix + (response if response else "Sorry, I didn't understand that.")

demo = gr.ChatInterface(
    fn=chat_interface,
    title="City Hospital Assistant",
    description="Ask about location, emergency, appointments, fees, or timings.",
    examples=["Where is the hospital?", "Emergency number", "Book appointment", "Doctor fees"]
)

demo.launch()


chatbot_app()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.



🏥 Welcome to Hospital Chatbot
Type 'sentiment' to analyze mood
Type 'exit' to quit

